In [ ]:
# === SSL Pretraining: Denoising Autoencoder (PyTorch) ===
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# 1) Numpy/Tensor 변환
Xtr = np.asarray(X_train, dtype=np.float32)
Xte = np.asarray(X_test, dtype=np.float32)

n_features = Xtr.shape[1]

# 2) 마스킹 노이즈 함수 (일부 피처를 0으로 가림)
def apply_masking(x, p=0.2):
    mask = (np.random.rand(*x.shape) > p).astype(np.float32)
    return x * mask

# 3) Autoencoder 정의
class TabularDAE(nn.Module):
    def __init__(self, in_dim, hidden=128, bottleneck=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, bottleneck),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck, hidden),
            nn.ReLU(),
            nn.Linear(hidden, in_dim),
        )
    def forward(self, x):
        z = self.encoder(x)
        recon = self.decoder(z)
        return recon, z

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dae = TabularDAE(n_features, hidden=128, bottleneck=32).to(device)

# 4) 학습 준비
epochs = 20
batch_size = 256
lr = 1e-3

optimizer = torch.optim.Adam(dae.parameters(), lr=lr)
criterion = nn.MSELoss()

# 5) DataLoader (마스킹 적용)
train_tensor = torch.from_numpy(Xtr)
train_ds = TensorDataset(train_tensor)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=False)

dae.train()
for ep in range(1, epochs+1):
    total_loss = 0.0
    for (xb,) in train_loader:
        xb = xb.to(device)
        # input corruption
        xb_masked = torch.from_numpy(apply_masking(xb.cpu().numpy(), p=0.2)).to(device)
        recon, _ = dae(xb_masked)
        loss = criterion(recon, xb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    print(f"[DAE][Epoch {ep:02d}] recon_loss={total_loss/len(train_ds):.6f}")

# 6) Encoder feature 추출
dae.eval()
with torch.no_grad():
    Z_train = dae.encoder(torch.from_numpy(Xtr).to(device)).cpu().numpy()
    Z_test  = dae.encoder(torch.from_numpy(Xte).to(device)).cpu().numpy()

print("Encoder features:", Z_train.shape, Z_test.shape)